In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
# relationship groups and expected KINSHIP coefficients (PropIBD expectation = 2 x kinship)
fo_rels = ['Parent Child', 'Full sibling', 'Half Sibling']
anc_rels = ['Grandparent', 'Great Grandparent', 'Great Great Grandparent', 'Great Great Great Grandparent']
full_cuz_rels = ['Full First Cousin', 'Full Second Cousin', 'Full Third Cousin', 'Full Fourth Cousin', 'Full Fifth Cousin']
half_cuz_rels = ['Half First Cousin', 'Half Second Cousin', 'Half Third Cousin', 'Half Forth Cousin', 'Half Fifth Cousin']

expected_kinship = {
    'Parent Child': 1/4, 'Full sibling': 1/4, 'Half Sibling': 1/8,
    'Grandparent': 1/8, 'Great Grandparent': 1/16, 'Great Great Grandparent': 1/32, 'Great Great Great Grandparent': 1/64,
    'Full First Cousin': 1/16, 'Full Second Cousin': 1/64, 'Full Third Cousin': 1/256, 'Full Fourth Cousin': 1/1024, 'Full Fifth Cousin': 1/4096,
    'Half First Cousin': 1/32, 'Half Second Cousin': 1/128, 'Half Third Cousin': 1/512, 'Half Forth Cousin': 1/2048, 'Half Fifth Cousin': 1/8192,
}
expected_propibd = {rel: 2 * k for rel, k in expected_kinship.items()}


def plot_rels(df, y, rels, expected, title, fname):
    """Violin plot of `y` by relationship, colored by superpopulation,
    with a red dashed expected line spanning each relationship's own category."""
    sub = df[df['RC'].isin(rels)]
    fig = px.violin(sub, y=y, x='RC', color='pop', hover_data=sub.columns,
                    template='simple_white', category_orders={'RC': rels})
    for i, rel in enumerate(rels):
        fig.add_shape(type='line', xref='x', x0=i-0.5, x1=i+0.5, yref='y',
                      y0=expected[rel], y1=expected[rel],
                      line=dict(color='red', dash='dash', width=2))
    fig.update_layout(xaxis_title=' ', yaxis_title=y, title=title)
    fig.write_image(f'images/{fname}.png', scale=2)
    fig.show()

# IBIS Kinship

In [ ]:
ibis_kinship = pd.read_csv('ibis_full_kinship_results.csv')
ibis_kinship['RC'] = ibis_kinship['RC'].str.strip()

ibis_kinship['RC'].value_counts()

## First Order Relationships

In [ ]:
plot_rels(ibis_kinship, 'Kinship_Coefficient', fo_rels, expected_kinship,
          'IBIS Kinship Coef, First Order, Full Genome', 'ibis_kinship_first_order_full_genome')

## Direct Ancestors

In [ ]:
plot_rels(ibis_kinship, 'Kinship_Coefficient', anc_rels, expected_kinship,
          'IBIS Kinship Coef, Direct Ancestors, Full Genome', 'ibis_kinship_direct_ancestors_full_genome')

## Full Cousins

In [ ]:
plot_rels(ibis_kinship, 'Kinship_Coefficient', full_cuz_rels, expected_kinship,
          'IBIS Kinship Coef, Full Cousins, Full Genome', 'ibis_kinship_cousin_full_genome')

## Half Cousins

In [ ]:
plot_rels(ibis_kinship, 'Kinship_Coefficient', half_cuz_rels, expected_kinship,
          'IBIS Kinship Coef, Half Cousins, Full Genome', 'ibis_kinship_half_rels_full_genome')

# IBIS Segments (PropIBD)

Segments are summed per pair and converted to PropIBD = (0.5 × IBD1 cM + IBD2 cM) / genome cM, the same scale as KING's PropIBD (2 × kinship). Pairs with no detected segments aren't in the file, so distant relationships only include pairs where IBIS found something.

Note: this IBIS run reported no IBD2 segments, so full siblings fall below their expected 0.5.

In [ ]:
# The segment file has one row per IBD segment, so collapse it to one row per pair.
# Read in chunks with only the needed columns to keep memory down (~2 GB file).
keys = ['pop', 'seed', 'Fam_ID', 'Individual1', 'Individual2']
usecols = keys + ['RC', 'Chrom', 'IBD_Type', 'Genetic_Start', 'Genetic_End', 'Genetic_Seg_Length']

pair_parts, span_parts = [], []
for chunk in pd.read_csv('ibis_full_segment_results.csv', usecols=usecols, chunksize=2_000_000):
    # make the pair order-invariant, in case (1, 4) ever shows up as (4, 1)
    a, b = chunk['Individual1'].to_numpy(), chunk['Individual2'].to_numpy()
    chunk['Individual1'], chunk['Individual2'] = np.minimum(a, b), np.maximum(a, b)

    is2 = chunk['IBD_Type'].eq('IBD2')
    chunk['ibd1_cM'] = chunk['Genetic_Seg_Length'].where(~is2, 0.0)
    chunk['ibd2_cM'] = chunk['Genetic_Seg_Length'].where(is2, 0.0)

    pair_parts.append(chunk.groupby(keys + ['RC'], sort=False)
                           .agg(n_seg=('Genetic_Seg_Length', 'size'),
                                ibd1_cM=('ibd1_cM', 'sum'),
                                ibd2_cM=('ibd2_cM', 'sum'),
                                max_seg_cM=('Genetic_Seg_Length', 'max')))
    span_parts.append(chunk.groupby('Chrom').agg(start=('Genetic_Start', 'min'),
                                                 end=('Genetic_End', 'max')))

# a pair can straddle two chunks, so sum again (max for the longest segment)
ibis_seg_pairs = (pd.concat(pair_parts)
                    .groupby(level=list(range(len(keys) + 1)))
                    .agg({'n_seg': 'sum', 'ibd1_cM': 'sum', 'ibd2_cM': 'sum', 'max_seg_cM': 'max'})
                    .reset_index())
ibis_seg_pairs['RC'] = ibis_seg_pairs['RC'].str.strip()

# genome length = observed genetic span of each autosome, summed
chrom_span = pd.concat(span_parts).groupby(level=0).agg({'start': 'min', 'end': 'max'})
genome_cM = (chrom_span['end'] - chrom_span['start']).sum()
print(f'{len(chrom_span)} chromosomes, genome length = {genome_cM:.1f} cM')

# PropIBD on the same scale as KING: IBD1 counts half, IBD2 counts fully (= 2 x kinship)
ibis_seg_pairs['total_cM'] = ibis_seg_pairs['ibd1_cM'] + ibis_seg_pairs['ibd2_cM']
ibis_seg_pairs['PropIBD'] = (0.5 * ibis_seg_pairs['ibd1_cM'] + ibis_seg_pairs['ibd2_cM']) / genome_cM
ibis_seg_pairs

In [ ]:
ibis_seg_pairs['RC'].value_counts()

## First Order Relationships

In [ ]:
plot_rels(ibis_seg_pairs, 'PropIBD', fo_rels, expected_propibd,
          'IBIS PropIBD, First Order, Full Genome', 'ibis_segment_first_order_full_genome')

## Direct Ancestors

In [ ]:
plot_rels(ibis_seg_pairs, 'PropIBD', anc_rels, expected_propibd,
          'IBIS PropIBD, Direct Ancestors, Full Genome', 'ibis_segment_direct_ancestors_full_genome')

## Full Cousins

In [ ]:
plot_rels(ibis_seg_pairs, 'PropIBD', full_cuz_rels, expected_propibd,
          'IBIS PropIBD, Full Cousins, Full Genome', 'ibis_segment_cousin_full_genome')

## Half Cousins

In [ ]:
plot_rels(ibis_seg_pairs, 'PropIBD', half_cuz_rels, expected_propibd,
          'IBIS PropIBD, Half Cousins, Full Genome', 'ibis_segment_half_rels_full_genome')